# TaniAdaptif Live Inference Sidang

Notebook Kaggle ini hanya menjalankan inference Stable Diffusion 1.5 dengan LoRA final.
Notebook tidak memuat training loop, optimizer, dataset training, atau checkpoint training.

- Run ID: `20260820T143551Z`
- LoRA: `multisource_multicrop_lora.safetensors`
- SHA-256: `0bda8908d577513110c22cf5ec1b8a7e008d2b26dd90525732a5c4e4b1bdd43a`
- Private API: FastAPI port `5000`
- Public transport: Cloudflare Quick Tunnel

Sebelum **Run All**, aktifkan GPU T4, Internet, input dataset LoRA private, dan Kaggle User Secret `INFERENCE_API_TOKEN`.

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import platform
import re
import subprocess
import sys
import time
import urllib.error
import urllib.parse
import urllib.request
from pathlib import Path

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

KAGGLE_ROOT = Path("/kaggle")
INPUT_ROOT = KAGGLE_ROOT / "input"
WORK_ROOT = KAGGLE_ROOT / "working" / "taniadaptif_live_inference"
GENERATED_DIR = WORK_ROOT / "generated"
RUNTIME_ROOT = WORK_ROOT / "runtime"
HF_HOME = WORK_ROOT / "huggingface_cache"
FASTAPI_HOST = "127.0.0.1"
FASTAPI_PORT = 5000

BASE_MODEL = "stable-diffusion-v1-5/stable-diffusion-v1-5"
LORA_FILENAME = "multisource_multicrop_lora.safetensors"
LORA_SHA256 = "0bda8908d577513110c22cf5ec1b8a7e008d2b26dd90525732a5c4e4b1bdd43a"
LORA_IDENTIFIER = "multisource_multicrop_lora_sd15_20260820T143551Z"

if not WORK_ROOT.parent.is_dir() or not INPUT_ROOT.is_dir():
    raise SystemExit("STOP: notebook ini hanya boleh dijalankan pada runtime Kaggle.")
WORK_ROOT.mkdir(parents=True, exist_ok=True)
GENERATED_DIR.mkdir(parents=True, exist_ok=True)
HF_HOME.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(HF_HOME)

import torch

if not torch.cuda.is_available() or torch.cuda.device_count() < 1:
    raise SystemExit("STOP: CUDA tidak tersedia. Aktifkan Kaggle GPU T4 lalu Run All kembali.")

gpu = torch.cuda.get_device_properties(0)
gpu_vram_gib = round(int(gpu.total_memory) / (1024**3), 2)
if int(gpu.total_memory) < 12 * 1024**3:
    raise SystemExit(f"STOP: VRAM GPU tidak cukup: {gpu_vram_gib} GiB")

print("GPU detected:", gpu.name)
print("CUDA runtime:", torch.version.cuda)
print("PyTorch:", torch.__version__)
print("VRAM GiB:", gpu_vram_gib)
subprocess.run(["nvidia-smi"], check=False)

In [ ]:
DEPENDENCIES = [
    "fastapi==0.141.1",
    "starlette==1.3.1",
    "uvicorn[standard]==0.52.1",
    "pydantic==2.13.4",
    "pydantic-settings==2.14.2",
    "Pillow==12.3.0",
    "diffusers==0.35.1",
    "transformers==4.57.6",
    "accelerate==1.14.0",
    "peft==0.20.0",
    "safetensors==0.8.0",
    "huggingface-hub==0.36.2",
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", *DEPENDENCIES],
    check=True,
)

CLOUDFLARED_PATH = WORK_ROOT / "cloudflared"
if not CLOUDFLARED_PATH.is_file():
    if platform.machine().lower() not in {"x86_64", "amd64"}:
        raise SystemExit(f"STOP: arsitektur cloudflared tidak didukung: {platform.machine()}")
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        CLOUDFLARED_PATH,
    )
    CLOUDFLARED_PATH.chmod(0o755)
subprocess.run([str(CLOUDFLARED_PATH), "--version"], check=True)
print("Runtime dependencies: PASS")

In [ ]:
from safetensors import safe_open


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


all_lora_matches = sorted(path for path in INPUT_ROOT.rglob(LORA_FILENAME) if path.is_file())
matching_lora = [path for path in all_lora_matches if sha256_file(path) == LORA_SHA256]
if len(matching_lora) != 1:
    observed = [{"path": str(path), "sha256": sha256_file(path)} for path in all_lora_matches]
    raise SystemExit(
        "STOP: harus ada tepat satu LoRA final dengan SHA yang benar. Observed: "
        + json.dumps(observed, indent=2)
    )

LORA_PATH = matching_lora[0]
if LORA_PATH.stat().st_size < 1024:
    raise SystemExit("STOP: LoRA final kosong atau terlalu kecil.")
with safe_open(LORA_PATH, framework="pt", device="cpu") as handle:
    tensor_keys = list(handle.keys())
    nonempty_tensors = sum(handle.get_tensor(key).numel() > 0 for key in tensor_keys)
    metadata_text = json.dumps(handle.metadata() or {}, sort_keys=True).casefold()
if len(tensor_keys) < 8 or nonempty_tensors < 8:
    raise SystemExit("STOP: LoRA final tidak memuat tensor adapter yang cukup.")
if any(marker in metadata_text for marker in ("placeholder", "dummy artifact", "synthetic artifact")):
    raise SystemExit("STOP: LoRA terdeteksi sebagai placeholder.")

print("LoRA path:", LORA_PATH)
print("LoRA bytes:", LORA_PATH.stat().st_size)
print("LoRA tensors:", len(tensor_keys))
print("LoRA SHA-256 PASS:", LORA_SHA256)

In [ ]:
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
try:
    INFERENCE_API_TOKEN = secrets.get_secret("INFERENCE_API_TOKEN").strip()
except Exception as exc:
    raise SystemExit(
        "STOP: buat Kaggle User Secret INFERENCE_API_TOKEN dengan nilai acak minimal 32 karakter."
    ) from exc

if len(INFERENCE_API_TOKEN) < 32 or INFERENCE_API_TOKEN.casefold().startswith("replace-with"):
    raise SystemExit("STOP: INFERENCE_API_TOKEN harus non-placeholder dan minimal 32 karakter.")

try:
    HF_TOKEN = secrets.get_secret("HF_TOKEN").strip()
except Exception:
    HF_TOKEN = ""
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN

print("Required secret loaded: INFERENCE_API_TOKEN")
print("Optional secret HF_TOKEN:", "configured" if HF_TOKEN else "not required/not configured")

In [ ]:
import base64
import io
import shutil
import zipfile

RUNTIME_ARCHIVE_SHA256 = 'f59278295985b98bd09aa9c35fba3fba1640b361288b40ac66d2a0ecb951712d'
RUNTIME_ARCHIVE_B64 = 'UEsDBBQAAAAIAAAAFF22N131KwAAACkAAAAdAAAAaW5mZXJlbmNlX2FwaS9hcHAvX19pbml0X18ucHlTUlIKSczLdExJLCjJTFPIzEtLLUrNS05VcAzwVChITM5OTE/VU1JS4gIAUEsDBBQAAAAIAAAAFF38BU2rGQAAABcAAAAhAAAAaW5mZXJlbmNlX2FwaS9hcHAvYXBpL19faW5pdF9fLnB5U1JS8ggJCVBwDPBUKMovLUkt1lNSUuICAFBLAwQUAAAACAAAABRdprobiE4FAABnEQAAHwAAAGluZmVyZW5jZV9hcGkvYXBwL2FwaS9yb3V0ZXMucHmtV1tv2zYUfvevIPQkA67StU2LCdCwAvUyY2sSOO72UBQEIx3bRGRJI6m0QZf/vkNSpCVKzjJgejDkc798PIfaivpAKN22qhVAKeGHphaKsKqqFVO8ruRs5mjyocp5PdtqlYapfclvnfw1/p1ZzpZJxRruOO+vV+u6VSAWZA1/tSCVfpENWoZOg1dbEFDlQFEvYU2T5LUA/Km2fOfs3IBSvNrJJ1VAiFpI77rhS004pXKoCyhlIvM9HJjXimcEnwuoQDAFLuaAaOO31F+BlWo/pH3UplfVth6SrzGQRk3aXgMreAVSHjXmpwKXIO55DjLZWUvYJtrRXBYXnnNjGf9qqzGx0duWlwUI39jV71cbulmvLi6Wa7q5+m15ebMgw/+z2c2n6+ur9Wb5gX5YfUTK6uryhmSk5FLFglU7iF+dv12Qd29/XJC3b+bz2WxWwJbkArAMVBh8xLLrcOp7vSBddOk4nzl58dMRXKmpoTWEjj09Rlea87NlYcFUHJ3tTceiBRFdramBQjbs5NxoGtQTHa3Vio3joaT1rh8s4j3H8lErjKF0GSRO24sKwCNXBZZiz9YPHiTVyiyq76LFgOHcZFKJOPD5OXKE6Mt8qFaADmVayfJGKqYwtKxZAUV2W9flWLMvMtK/ByGxaZlrrsYc7YhHyYkueV50higpHnoVCLo2cXJCSZl9P3/5OiXfbbBROj5uj49WLey6cR47Q6mfXQYGIytpz3XH6mHA03ow4JIaFyhmyutlPkc27y892S3BsexV0kGlXYiJRQ3NMVM0inmHkBtFPY0661479THiuAQSYQw07MkIlb08noPHnvizkdjTeRKDKCc1BIduLBWFdYaebidEx5vbhC/rCp7EanR2/8OZCeEFDth6PFhG6yBEmY1fK9v5MlJIwyaOJIZNvGXeuT96R9qwPmUtGMUGVYpvuW6fUwgYp7rtxB1lKCfbRm8SKGgu6kZmZikMF0jQLyX4bocDRtV3UMksWD7DIHiJYAwUvmtHKTH/yLYW9m1BNBk34OROS7iCA57Lx1PBC7usTf1k9jlCNLW5vjRhXnWl4JvCtkcFF5Arandp9OWUsa+8UPtsamue0tgD3+3VM1QQTawtMQQm2AEQoViPgYB+Igl4TlLiW+e0NH0xIa6gkSj/6nyCuWt5wfSVQuasBJR6l0yJmZynfBrGhIJNeUrDciZUDuwbNgrhALKv17sm9SQm9BU/AB5rLAP2tDhlI5Aa2nk8OSmaWnajorMF40ER3jDDOeE0j4e9YQ967KXhjZX8/dRtUz8dolMy4JjxE4aRhko4FXC3dL6THhFPW/fP7CHo8UIbIwG0OCGtxHjRIQhQmH1lXPnl6kvTRbXoGTuuUPiWQ+O/ZZKNbaX5TECiZgfOGMcN4L4k4hFkess2O3/5Zowpw4kulpfL9fsNHle6wYN79WkTjUUPuH/YDqX/ZHeqJQ0cblumWEXuuWxZifwSviIEcbgrjLVkD6xiVTJhSSP8gd2WkG1EC0P+nJgvAcw0LMq6rTS0/69ivH5GMX55v/p9+eGpWvxhc1c4Y+5IwfC7kxRc14XUJez9FiKSIYlX/L+WQz8FKMZLHJTHq0rqceVI/oVWOFgfTxbV07e8BC2KSNWfx7HFbcIPmBfV38/zRLOPtzvDaEWpb4zmrmKPCVIobrE4cggvqBGVOD60mcw5mo8+LsJzHFz02hw//OREUY4nJzu+DkX0Ms26lHDPsAIxaS9DmhMFK93nlvm3KQHjr1el0KleT05ivKqCEe3ms1M4wZ64Zfq8xncll2qY+mjqdx2ws3/2D1BLAwQUAAAACAAAABRdjgGU/y4AAAAsAAAAIgAAAGluZmVyZW5jZV9hcGkvYXBwL2NvcmUvX19pbml0X18ucHlTUlJyzi9KVUjOz0vLTC8tSizJzM9TSMsvUsjMS0stSs1LTlVwDPDUU1JS4gIAUEsDBBQAAAAIAAAAFF16XzjL9wMAACEKAAAgAAAAaW5mZXJlbmNlX2FwaS9hcHAvY29yZS9jb25maWcucHmVVm1z2jgQ/s6v0OiT6QANNOlL5rgpDU7quQQ4Qzo308lohC2DJkLySTJp/v2tjG1sXtqrP4Ct3X12n9VqV4lWG0RIktlMM0IQ36RKW0SlVJZarqRptRKnk2QyskoJU6oInZGIRmu2k6fUrgVfltIZfBaW6WtMpeVRKbrlTMQdtFExE2RLBY+pVbqpSwyzlstV5e0LNWxerHVQ+XajZMJXYx7ZVqsVCWpMJfLqFu3rFoKHpimRdMOukbEaDRFeUMlHMU0tT1AgE6aZjBgazQJc6W+ZNpCGyuSi1+9d7MSpVlseM13KcmJezBKaCTvEGxU94w4qGAIGgTdqhjiY3PqhP7nxySycfgvGfojbOeASQiZ5XpqQudA9FbaxdClYN+ZJkrnwutt+9+rtyVXcqcyPQ/kymvvkYTr27wu1XSBCaUrcjp6hdpLW/TQckdlo8RXXQMyaDq7e/zbM/OsIzOpAkGgojISfTbdUkp0HhCxPFsFtUOVas38zDiW/SrNrtITCPkK8pcKwU4Ch//djEPrkbvZ4AEZTTqx6ZvJ3IfclUYIvpn/5kwK+Bvv/87iHhHpuwK2YZJpaFpOYQzbdUT1fak7qYV4eDkfxbQWA26c83/kTPxwt/DEZByGulVWktCFK8xWX5gyVNye53EzDOZmGwV0wmTdJOCXLN0xlFnpGpGQMyIlQ1J6n9Omid/GTuIPphCyCB3/6uCBz/2Y6Gc/rJGqON/QH0cxqzsAplz9x2e+A3RC8CjYc/ML3w+gfqIJFGPgNvwUUsGRx01vp5HJQ90L6lx/I5cd35P3lh1Mex/7t6PHeUfTHuOnihcfu6O98XPUHDeGa8dXa1qW5eNfOo7whw/pxh94nhcktSTjEiHvwWutPpYBApakYzIc4s0n3Y13lh9V0iPlKKs1q66lKMwElSZaveYsfLnTGypaW/32Gdp0ybV9LNvVyJIIb6xkmkjbq/onc13eoz6frygFMCCfu1Wu4Byo89dpoCIPhDd4r73oCjFSJvoPgqXW4yC3bVNaJ0sgtQEpP+UgFtx7u4LaLoW74VBA7mKSe+x5imlim9xtbVkCtSe354nK7ahxSQSO2VgIGHOxnHldlWUbQi2BiJaADrzB8tDUv3LULzXLrrvsqQqjn8Khdwn0jRp4oQtr7aaM/0LsBgvzUomkf5Jlyw9A3KjLmaw30G1L3nGqFaJMZi5YMUQSDo1snC20Etgm52BG0EcEoaEIU+Bg4WlNNI8izQS9roFGQycCSm3IqxE3D9mExOMpwffm8v1C57VoxW12CvHyTyj3a0a8uSPuz5rUbkl6jz/c2z/DrpRTauDW74wGnCQqdqOf8sxxlRVA7kNZ/UEsDBBQAAAAIAAAAFF1QZN3z1AAAANYBAAAgAAAAaW5mZXJlbmNlX2FwaS9hcHAvY29yZS9lcnJvcnMucHlljzFuwzAMRXedgqNdtDmAgRbIkI69gqDYVCBAlgSSBuLbl05VK241kf99kfye8gzW+kUWQmshzCWTgEspi5OQExvjN4+sJaTbLz+n1RgzRscM5xIuRJm6y33Esv3pBwP6JvQ6OqQg1nYPZXuM0b/u3UsrWRcubMc84QAhSSM/Egs1aUZmd/urEgqt7hpVv+Yc4R0+XWRshgnFhcjDFkDpV04V9vD28WiHds9SkLr+tEeoO/tDlNPT2TrxqTvaKv8P6lRltTriPZIa9vpoqaHUUCvzDVBLAwQUAAAACAAAABRdNRRT2jMGAADfEgAAGQAAAGluZmVyZW5jZV9hcGkvYXBwL21haW4ucHmtGFlv2zb43b+CI1BA6mz1WFBgxjQsaJ3WW5d0jlsMMwyBkSibi66RVBKv63/fx0un3RbD9JDw+O6bTnmZoyhKa1lzGkWI5VXJJSJFUUoiWVmIycSe7XMSu3VW7nas2Lktp24lWd6s65olk0mqOKRESFIxR/4CtufvllO0on/VVMgeUEAfYlpp3g7egn0gGUu0VAvOS97HylmSZPSecBrEJW9QX16trn9trvoonIoKuNAG+Ofrq8uVPeyDCmWNOGVZC3ytjy7UkdWSFSnltIhppFBIVQWaS1nLFivmlEga6UN+Cg0UUFoUKds13KiUYHExRTsqI2F3nyVAlZEaxucV61htjCIov2MxFcGOFpRrK0f2zJF43dxcm4vJxAZCcEMEi19qib2M3tEsdDfLy4urKUpLnhMZ4kceEbEKEl+gRwayIHaXUyHIDtbY13QpR6GLNJBKvtVnHpakYCQhECJp0OgBOJNJQlMUCZKCfU3ERCzx7khW0zkSkqN/0GVZUB/NflTb+QTBx1KkISDmE5TRwsD76IcQvTjTh2DKtM4ykD/eexxvzmd/kNnfT2ffB9Fs+y2eGnzfkFMfp5BNhTmedA5UQgTqz5nnB3v6sJk/e7F1YmtvRTdlcvAsjtZg7oJ/qk8fm39xmRiVzNZarnMCHPmB3GRwdlOWmTlMqCQsE3NU3vxJYwnWVeaYTrRBEhbLjcK3t1ujjxZrPr4F5I+NwljJg+darGl7asWCC7vq3DUCwm2zNvefnF+svIgJBOVIC9saWQu2wRYGK4HsumvyjoyijiG8BfC7IJnoy+KCBS4h0IiU3LOHOvPptAcEu7q4Lcr7AvsdMloioKD/O1WMd23WQ54Z57r8nTd5bWOz8YmB0lk2HyfeEFp70FZVYyICWXZHm0IBkM2y5L0a4vnHEFwZoEmUMB7kt/DXq6CEFlKEa16DSegDA3uUt3o7IGKkDJ0KiudICW/A0pKoKsCzuniSyYyGQ9kAJlJlA1KPcgEUj0LYOygMmm6WlfcROCIBFRhEAHAZYqnGEZWcQcWJMlAOfROiDX6Mt06ygCRJ1HYar3F+v820QWG4GpJiJOWQ3xCvI204OhkC51Tuy0SEG/x6sVYx+u7qeo23Q7A9JQkYBsCgXEugNVsfKqrgf5/ZUjNbvjL7pauus3V5SwtHzFr0J2WQjjHwXsoKWy+KQxEjFfoub2wh90Z1DcUgWFTQB9kroZ3sC9rcA6eNCryDtZqpVuH1lfH9hjAQ5KryqR4UVJSnUVzWYAfudWDqqsoYRL5UWgPwCQ5D84DNcEtFI0d3amYBEqoKDwNejQeGh68bjZqwICZylWaQdTtg2UbYWLBp7+4k7RaslQ2K6xBe6ci4nges3qb3Gc1rngUVkXuVEfgJ2CGTe6whVGXuqDrvCeUmLDBAd7bqa2XdImsRqQYSnj19Nh0BxCZYw1GfHH682y7HVIA+hpFksVpcvlxE5+/Xb6LV4rf3y9UCYv4oju1dIf5F556A3EMZOZCCFO0YBfGUkFukTRCcINS0uXDQgNzn949ad1EAP2lYck+YbJPIpUOLndSm7KpAhlhPvCOxj2YmM6boO38yZOLiftPPqm0nM4Z52tAwg5ya08q+v/AjmPmEdXwIKyemWreEYDcwp2Np6t3xOxeuw1urTyfa+hBOhuNUh0oeyyw7dzhWnUrZPGuiPeRNBlZ3I/mwZKoUNGHuIMdF085lDQk1AHRTbDSNns6/bu5psqft81VZeDIDOxzGpNXnUs3AjIbGjkY2kQzgYH7sOdSMhBbQ7gZp1nXj59x1/BU6dN5dcz3w4bC3jZ154pmrJTvtXzclw6jS0+vjyBg4K2NNF4ZUNcx7TNLcV28zpFZQzWDUFjXdKEC83Y7N2RnpLWQudvgYoFRDRQOldwOwT72dFkIBKymMt8z7tdOXt/8tps+eP//foxh/OH+7fHW+Xl5dRovV6mp1pOY3jeMdOWQlSRCU3JwVkkDj+EK7+GKrcHH9lRGtJzVV61s98RPzm8aTZtDvyNH5bcODwR8efCU/jMbX3hOhw1SN5iFub1kOZnBlvJnyoSfEWZ2430K83i8jw1FpOnha2InOxgFQg4eWeTl0nlr+5F9QSwMEFAAAAAgAAAAUXXX7C14sAAAAKgAAACQAAABpbmZlcmVuY2VfYXBpL2FwcC9tb2RlbHMvX19pbml0X18ucHlTUlIKqExJzCvJTFYoTs5IzU0sVkjLL1LIzEtLLUrNS05VcAzw1FNSUuICAFBLAwQUAAAACAAAABRdITkg2K8EAAD7DQAAIwAAAGluZmVyZW5jZV9hcGkvYXBwL21vZGVscy9zY2hlbWFzLnB5xVdtb9s2EP6uX8EJGOBgiuFkeekCeGjWbV2BbgiKYQMWBMRZOslsKFIjKbcu9uN3JCXLke0k7T7MX2xR99z7PTyXRteM87J1rUHOmagbbRwDpbQDJ7SySdKdGUxKL+3WjVBVL3mt1hl7KxwakEkUaNYFKCfyXuQHsPirLlBm7JVWpah+FLnL2M8CZZGx0n/xFUhRgNMmSV4Z3bwp2LzXeps2UIg0Y+l7qFpV+V85LCAc3UMOquIOFCzTu+QtqOL3dYPbaAsf6B3JOqxAggowXLSK51A3rSHYn0Cy1ysQEhZCCrfexuftfdtEvFmAAxsUtIbsEvTG4Ero1nqvv8Rn/6ytDrqSJJdgLbtuRMjXZJO5o6uE0af2v3kekkjGhmxOGqMd5g4LrqBG20COdj45OtrofI2KHPMVvQFDIuSmnfSGvlR9xvCjMzBPS20WokjJnNdjEYsrJpRj/7DftPLVCMWeFFhCK93cH2aswvksYxLnp/zk7JKfvfiWX5xdHgUVBv9u0TouSJF15nFFtVBcoqrccn5CT/Cxf7o4i9qsw8ZGj8YaTs+DIyfBkfNZlK9aakaVI7c5SLxipdSwC72cdthpDOPkfNrhP4jCLffbOz85DaDT84sAurx4ETFLFNXSfQYooF6OxmeSBtu+raLCNGp/GbqAyr7URTgg1awQNSpLLcHrlnK9QPqWTjQSuS6pFpNc2oyR7haDX0fs+Hv/HbvFf0QZX7Ov2cUZ+2rOZsO7UEYQFtkfXuQnY7x7G5vM22QLZMB6q0yXpKbzOHYB0ZKKJg7FO3TKY6EOUptYLZT4IMKhz0Kgw+OeeIVlxJCxKYlzwoPBadlKWYPLlxOT3l4f/wXHn2bH30358d03aWfn6KkMDa5SK68ZjaMDoZhWck21d35wM6baehF+FJqotAC7zFirCjrKtcGDKXxIBvgumprsI4fOz5x47YpFTg4HxKAFpzuAEtazbex5z6Ectkj0iu3wapBsOsrkUfc2g0ZNCG5Jumxb12DWsTD9SGzN+umDWT+fzQ7PxEONjzWK0qYm4CfkI9C4VfoeGcq5gfrLK2Xp9L0WahIQU9tQ+J6PtzrJ98yAeaotRu7E8fEqaIQWVJT73aIPyjeVv6ELunGfUf8mAJ5Zg5PZrGNARXetEyvk+/A9raXpHvSBEkY9z6ucnyBh6LKKoP+7ctGLzy3Y/jSM8vq8fIxAT+ejc+dQJpKkb6AbWNPV6LM2ailizr2ttoeCbENbJo53EdvmtGPQpb3QWu5bCXbYqSPrcC5qqJC3Rg6i8WgbvFlT4p2/mQHu6I7iFol5C9td/sN+NOAp30BlgSu6SHN3S8eZX4fvxn3OS8ipeOu5Fxs2sl8QpFsejJ/W75asbxZKfU8bYjeRK0FMPzhSEIHmuOVY2ON8YXyAmwSuaKopvCjXe/EOaVFVlOlnO2IIsfYLBvUyjw//0TFSYju/hkUv1LL3Mjj0RpX6kJcL2pT5qD5SG6CCI/0XoTnacmvXUds2/q8KUYbvKIpWChsqGiNzRlQVca/T97S8bNd7I9IISel4huBgqm9o7/aOyUGsWybDa2rW8et+b3z4vm+9ZsPn4y5N/gVQSwMEFAAAAAgAAAAUXZDZMzAkAAAAIgAAACcAAABpbmZlcmVuY2VfYXBpL2FwcC9wcm92aWRlcnMvX19pbml0X18ucHlTUlLyzE1MT1VIT81LLUosyczPUygoyi/LTEktKtZTUlLiAgBQSwMEFAAAAAgAAAAUXaA6DlVWAQAA0wIAACMAAABpbmZlcmVuY2VfYXBpL2FwcC9wcm92aWRlcnMvYmFzZS5weX1Ry2rDMBC86ytETgm4+QBDStpSSi6l9FqK2FirWKBXV+tA/r6yHTshtNFBsKPR7uyMoeilUqbjjlApaX2KxBJCiAxsY8hCmJ4D+2Z6fHp+qUqdmaBhj9xGPXI0MDQOcsY8cWeoksaiOxP5lGw4zP3C6TzEBoOEoUEFya4hpXVGOtoG8zqV98Rq31mnkaavH4QJCPUbBqRB7yf+dJhZCLGdZ4vhltek3DmuhSzHejigSsBtLctKV5jVFyQjlsoGHqrD3Emx9agyNjHoXEvjIowUHzW6y/9iE/R6aqltw18Frvq9v+Vm9GWp0UDRpEzxNNJp09NWZYtR+q4XdNYfaVkSWI3qiy1HWwxRATwO40rHxR4yLsRA2N4E1WNllmwRHLfLjM6s5MPjjayxeX8IbEb5Hnnnk0OPgVG/EkW63/7sEA4DKkljKvX/gc2cyfZB1d+R3df1C1BLAwQUAAAACAAAABRdbYSASN8EAAAoEAAAMQAAAGluZmVyZW5jZV9hcGkvYXBwL3Byb3ZpZGVycy9kaWZmdXNlcnNfcHJvdmlkZXIucHmtV1GL3DYQft9fIfzkhY3bJCQPCy6EJg0HbTnS5ikEobXGXvVs2ZXkbcL1/ntHkmVLPm+uhS7BZ8nfjEbfzHxSatV3hNJ6NKMCSonohl4ZwqTsDTOil3q3m+aM6GBXW/zAzLkVpwC+xeHOfxGyBgWyAsoGUbBhKKpeAT5kLZqA/w2MEbLR10wG1V8EB6WLE9MQrN6DBOVC+gB6bM2B3HSsgWm6V097a3vF6IW1gjs3wfHPOP1GGVGzytzIuj+QCQPUWbDp27UFNKiLqEDblbrB0NMoWlxvZkfBwBTwOP4/R9Bmt9tVLdOavBV1PaIXnW4oT4f7447gL2yHStYBKUnGg3W2cwAONSZUSGEozTW09YHoifDjTP2ePPuB/NpL8E7tz0KLgETH4TUF9KMZRkO5UBGkaHyQwO2HawZFd4fP3JIhjS5/VyMcCHwR2tD+zg33qekgBrtDG2b6gYMl3G4eqxSTxDjwLIXYSQpK9epItFHkb+dm01uS5OOjcrhiatTXhbvZlwuGdj2HNl92A18qGAx55/7YymPazm3Yb215Y0+We6Ny9LGPkh6tbfHXcpzsFz1tV3v+eP053Q5oReDwFEif2YtXrxfY/jp/QWV6VZ2TD67t5iqfRcSwUwu+dZDSWySuFRFrE+c3Dv3OkbbFumICJebDKK26OViezf1IlJ9HegeQHLtegCZYvwTrDpVAG9a2wIts74NE77vZu6hXhChse4Ei2wwj6it3Ptxui2rkrBCasgsTrd1Wvn86yve3H4nAEL1XTk6jIT9+fPvGTlrXszMMb3etf+zKmQ31aiAEWlw6q4Zx6TBuvroy9UY11p15/nrecPA+u3ceIuzLF5t9fiWjhaWWDgoMsiCBf6sw7XnhOyCtTLc4dWGX7rlVkXEs83th+jza1Qa6AGnjpswY1DUMnOpWVBhOHtE+dwz6tsdl/rgTC/t5y79rawf9C0RzNnrFAArB7L7w6rpPt+/t3IlRLlA7jHmYheQMrJ1CdBrCRWU+4TIH0p/+gMp8XooTszIqSe6T5bJwRmXHaRvxmZWGlnleA9KPVhCX0CDzxyhPU6VbjVuZxEf9YpQq3xPWXro2TQv/ba741K8r9w2vCpju5eIwiPkCe1hyEE7U6QRX/spwvH6bmDFUcHfkudStL03HR/IUiEwPCpfbx6KzPobwX/aLzU4g0ycpUZxE15fyNsiXlRD0XdhHdFw24cozS8xyJ/IFUsYtWXRMjqylGlAcJhIKO9hHMeBNKm7rtIX8xa0Mtn6YZk9CgzRegK6wq/mV0djR5b6oDQx6NnSjFN6MWK4OWbEWZmQ6vepswc15RrpRCji71p8RfrhaNpBbzm+RLBSOO/3p+88pm1hmSGidaf78Fb1fau8hW+Em1VvfHr9D2/vg6aEYZLMyLDS7QL74iEpqUp11cadJXSxLK5GRo8MGTvAyvKwvNcDLuKo2ybOyb6sYixD/r8Mxzf0oeR5VN3nma/5AXq4CcPJW/ruDrAPDUNJYmSqu/f0X1XX4JWdosQw2kE9rtINVqh8WV4UdbqCMEk2DUZn+DmQET+Y37JzI4n5Qjmqx7DK9by7fr3n4v2U9UI/NH21mSw0ccqUXkck3lcTZuvaOLDba3eF8k0fAra53SLzqdX11h9CfGG4vRTzEl4N/AFBLAwQUAAAACAAAABRdbZh2DWEFAABCEAAALgAAAGluZmVyZW5jZV9hcGkvYXBwL3Byb3ZpZGVycy9sb3JhX3ZhbGlkYXRpb24ucHm1V21v2zYQ/u5fwemTtClCkqVZZkzBvDVFOnRtEWfAhqIQaOkUs5VJlaSSuEX/+44vkign7ZqiMxDEJI939zz3wnMtxYYURd3pTkJRELZphdSEci401UxwNZv5vTVV64at+uUbJfisNtcrqmnZUKVADfdVxUqdjkdOsqXaqOilXuJyNvtz8XexXDw5uzx7vnxxsSzOzxaPzy6K3/65PFuSnBwck+/Jwf7hkf83m81+HdTGqPY98PxSdpDM7BZ5JiRdSM1qWuqnvBbzGcGPMT0nSku7Uuw9FKutBjUnjGu3t6aHj45HGQ1cCVmUouN6lDKmw5tuE2qiRWEwxwqaOiF7p8SsXqGylIjVGyj1a+eI+UhAurlnyd1AWEZL4bwoatZA7Hw2LFmFqMupqNgVKI3c+JBk7lKc2NMbptcWbiZa4HEkV1GCplCYVw2MTtRCknLd8beIgzANMm7oZlXRuZfMJNAqDphPUrKKomTUMPqSdS0SA7HV59zwGP35Gm7dt3hAyoXc0MYEwrt/TZsOLP9TuINghZCtUIZnrI2TrBE36LczyGrSIN5ROiHf5eT4iCBOyrfx4Ha5phJzAyRq1gZ8tH9w+OPRo+OfTn6mqxKdizw5vRzKjGqtnoAFSZkCcoFZwjZwJqWQcfTsxcWiWJ4vEBbZdBirFRBKSrFpG9BArNslxWtopmtbv8ALe+aGwxhNeAzsO/5QiBnOiwbzvaA+4WNJb4oh11MCty2mHlRFkN2W3furBDk0nPRKep6/AOzLxeU5YQrdfdcxiaEyDOo1kMesrjsFUpFWimtWgeyReWM7Lj7Apif4YVZ3zGFK3c3EHZlk6CAobcpxYDnJUBSrxZjCdJSgRHMN8QSfJZIpV9L/Aet3wWt21Rkkz8TFgvRhJTdYwEZZjd2oMtVMEfNV11BJjN6AUhe3rq7ZbWayqhZNFdtSiDJFa3BtTUVf6ciYzaG2iRNjc0W6fBpRLHz8V5iz3tNA7pecnDzMoSUav/TGB+cwE2DT6q0pKy07XmJ9VMYtqxolOtoEcd/ttQOHU0mkbreMvo67vr4rAS6aG6rLNQlyefD1S5v4Gns0SEsrIsIHKTMPrWM1Dvv4ieneuCskyudRw7RuIEoxCFccqvwJbRQkg1rkIJ60+dAOBmt/cohsh+en5HNP+udvBjmxR04G2Z1H56EJ4myY5+EKacUsYdy2zz5jzUfL7fweyMiqmXTwpaHVlNHA8SSroBQVxFGn672TKBnVwm0JrSbxX5wZicdWzrqcOsV/LF88D3ZtlPHSNwFc2jnOlKvJIlsKxA5iaCBsUUwxjjXKS/CoUju9PLBZ3ePA0C24H4KGBB/nKGR4GnX8C7gNZjEj+uEtbOfuhbTNHpepX+Ir7e5lOM1sFDY9hIfntvcVxQY0tVaL6GMI3uv+Bu2nFFxTZBLV9lr7DOOC285UjEhcDdkHy02anG4AiQdV4gOohR07ejUe0Tys0J3IjRfvRO8TqOroqasDh8nZChyYkw+Bax/DahF1rUAbGKN4dgU6jizF/ji4gI2zhXvE7X4gV+ntvXJ2P5CzfhejG9N2FdDiRVLSMKWTiRQWsx0ZvUhC8pwc3pGgTRMH+kwkUtNpExs7szSB6nXcub5v+qU/fbX/Olwd2NVYCWO724HZkxe4Ybc8qMHNT3EQ+GxErc+nmIETBFbjPS74TJsSLmSw6dzzW2FGmpilbuT0p3brKxOzL+DPpCX6GpB7GvA+tXmnHH/AH5pjU9itVaTqf2kPNvZDS/ST/u5kPkbVzAM5sumGlXSsraF95uPXNCw9HC3yyVAznoY/c3NTDR51oH/M0Hz86o6T2b9QSwMEFAAAAAgAAAAUXeGiURhZBAAALg0AACMAAABpbmZlcmVuY2VfYXBpL2FwcC9wcm92aWRlcnMvbW9jay5weaVWX2+kNhB/309h8QQ6QpJNbi9diUqtqp5OaqvT9XQvUYQcGMANGGqb7EbpfveODQZMWFWn8gDY85vx/PfkoqlJkuSd6gQkCWF12whFKOeNooo1XG42w15JZVmxR7tUcFQHQdtxzWrY5FpcS5UGWlmfcbnpKZ8//WZ3P9W0gLD//CLoYUAwnoMAnkJCWxbRto3SRgC+eM4Ky/snKMV4Ic+xtKJ5ZhkIGT1SCZbrI3AQxqYvILtKDYcP2404J02CeGYpSC22blXy2LEKhY/mCWipgGwu/u8OpNpsNmlFpSS/N+mTe5TvLoP9huBj1U44rYHExKuR0dsYWgY5holxppLEl1DlIZGDF/ajPwJy8SP5o+HQy9OPhkYWiTLtrwtoOtV2KsmYmEGiotcPMk04xxDVT/j2tQu4kvFX0WFU4cikSponswwmC0qglSqN/kbXjKXqXioRkubxL0jVw6S4AMxITl7HDf141kXevtfDcVnoYjPQYUOkl7adtyDWTQZVUjU0gwwhRm0XIYDKhiNNO3SinSZrrH+GeIg+7PvzGTFiEpahAUoYJyzzchY8RTHBYlNakX75wUhjOoNQjgkYeiKxG754e9iSTVeoZZzF/pLk3quVc4paXnhu2JODYArZK5pC2egq8CeB44HhqFywDOfSVt/x+SQrRufMRAfhCo5lsf1xyRIgiwddIr1wycWoQ6J9mkjA7pLJWDQdz/yZr8lFH4GQ3CwUMNkTm/q8mDljkWQ1KJpRRWM3iU12fUciD9loY4kc02IFmYqmnTCRXq6glGBFgcep5gn4DO7sr/D1PXDG0G+sIDkU6ORnDOCSZUFZ4T2wTJUzDrNewZXAinIuut9YQTKZmG66VuqDgyVQkZZJXw8I/JVWcoE8TctZV5tK7/81AvxOtZ+xAulYpMO1G8mSbt/v3ILx39iBBTyJPv3z6oZptqHrYrY0Hj4dX11HnjzngCDCixFT3/c6lV/ceVN5B8hw7FX2g/v99e5hWfm5qZbktQed5tfa26bidJ1wpXntzVQRzg75T5e/6ZzG8f32ysX5SNOnwrQEjIG/3d6GZHt7ha/tbjI7x9lkAt3cheSH65DsZgiapngvauoOKdc3VxqyaMdINfMAFsbB9758/NkLie+m/iLBg3Cm3yQtwznKCtMzVaRffRudQDUVBeMIq+nR12Y5J5HLS3J954qMBF7NlBcVuPl234sKif26ki7e7PfKj4QHt7iw8CrGIZ6c6tKN1Ph2XoLfoSV5R663VtVhsa4v/jjID9uFojmrqrgP7Fll9HTs+6MM7edxcXOF8fO+Us5+ymirWG4mRPKNyY5WGHsj3zuUWBXe0GaM/Thst6AzzY7ekX75i05s/bTFOrybyvDFRNwaf/V+nsNE+x0nX3vE3jH3nD0vaIVmHBSewhY47C/kXYz4Ffe4rcyRfSZj8Gd3tbiLc8/c925jIzgcTJOMtxK/tSxb1GUk6TPMp5DNv1BLAwQUAAAACAAAABRdfDlPijAAAAAuAAAAJgAAAGluZmVyZW5jZV9hcGkvYXBwL3NlcnZpY2VzL19faW5pdF9fLnB5U1JSciwoyMlMTizJzM9TKE4tKstMTi1WSMsvUsjMS0stSs1LTlVwDPDUU1JS4gIAUEsDBBQAAAAIAAAAFF1/+OlV/AQAAFwRAAAwAAAAaW5mZXJlbmNlX2FwaS9hcHAvc2VydmljZXMvZ2VuZXJhdGlvbl9zZXJ2aWNlLnB5vVfdb6Q2EH/nr3CRKkFKafO67falX6p0raK73FNVWV4Ydt2CTW2TvSiX/71jgwEvsEleik6XxZ75zfg3Hx4qJRtCadWZTgGlhDetVIYwIaRhhkuho8iv6UdRcOlfa3k8cnGMKotQMgOGN+D1P97/mI2LvUjLzKnmBy9xh69Rv8NFBQpEAZS1PGdtmxdSAf4nKn708h/AGDSnt1QaWUKtc12coGHaa/0KAhS6cccea8nKLeVWyQdegtL5gWm4UEYS3oPuapOR3xp2hGFZqpfRSl5VncZf1K957J/8zlshG1n840F+x9+v00dLD7wAbYGa1tBDx+uZN62ClmH8j+OJqYJ/O9AmimycUXLvA54fwbxza0lsmOCsZK3hVT4a9cbiNIqiEirijI0MJHqI5G6MaUq+/uGC211E8BlZ2xOvNVKR1/KMaKkT5NVMdk9iy1LcY9hHAaa3WOFrdGYdZozfEmsjgBeAinHMp/edsHXws1IoUMUfhe5ayzuUo7kdeVqc8NkxWNRM61kufujZ7R2y9FLKBTeUoum6ysiS3mxmJfSWfCZ/SAFIr/3jwmB/TIe1kLlHnEUhFJiFafyJ4BtxT0PledLVNrn3vtPk7/A1uRQ/WCEoKasMKGp5lZ1BpV9YrSEaWTkBq83JceLOVfLC/KmNyog8/A2F+WsR0eAo+aCfTogKWMkFaP0q0F7fUbYK6+X8DrXwjyh/kLJOejFbaknsOhu17QvKOE0v3X4aF+xzc9OrZsFq7MDj3aU1JkqCjf4as0sgLQUiJcG627vzsbeqJUHl79BJbRh2GK6J7Shc4cYBKuzvaFyaE0pP8c/jBShW5BXnFuJ4BQCZkzf4mwaS6XSo5z6+LuNclAdvIAnSLpux7vri7vJqWUhQXu6IzQ234dLl8kKZsuX6MXeB9ys9ZaIeeR4giIMgjPgSscEeYqB9XPJ4nonuEihdFW/dB8nwN7NkMTwD1QDlPugT+XxrwmfGAN492tfEKD6z0rBPaMkoji5+RW5HXeyBhoI96y44edjAosmWi+eZYwWuNpmQUUxH7x3enUiwOEJym00eoyvpbpFsRj0uF/sMsNG1jezM+Dg45faFoq1l6QRuo6SR1JxsiSZh9/DJmY3BymbplmabwEMO7DeJHwQwZDh2lXodKV1d7Ql+YDW3Ax9FlBaB7LiX9ETk3F46bmUdYRBrwOAwYVjetWVQgZfP0+aOfWI7w1HXNbFPhSeetra5chi1VAwpBUy1ioNaAF3sv4BmCwEhhnPatxcUfKCxERhU9JN0LuQ5wdk6zbmWmEkNM8mVqDsohRFtGhBlH2gtO4Wz2W7MoHxV4EXQPukslSHWtL4N8fyG7BpuuZ66Zb//VEA7Vdh9n8WuNayX5vUZ4l51sKrWD8G560DbeRlPBTV2XV+7+y/1rFTxLd7mJyz68c4WrLlC6gT+luJ1l8kWrUGnxY8qXF5ndWrPSCJKXePwzJTAEnodixXj9WxKXpDoWzT+/Ob/p9Q+gwMvCryppeI44O+j70eAdeqdieCa0TVAm3yb35Ibr5tON2M/PcwChv/CWcJ9IpBFEOyoMH1zrLb7/gNEsbN7ddPPyjfFoKGkNFujAJZmyRV2Ey3rB5hPy8zN1HfuchnspCty9kMOt3LblYQhX+wDq3hkO/U6Ca5phcdL0jcMWX1PsiOWwHnBUTHgE5ypeMUKN1j9B1BLAwQUAAAACAAAABRdsmDg+5EFAADIEAAALAAAAGluZmVyZW5jZV9hcGkvYXBwL3NlcnZpY2VzL3Byb21wdF9idWlsZGVyLnB55Vdbb9s2FH7Xr+D0JBeK0T3Om9cFbZYF6xLDdbuHICBoiZK50KTGS9zU83/fISlTki9tMfRtQRCH5/Lx8Fw+0pWSa4RxZY1VFGPE1o1UBhEhpCGGSaGTpJUpmlTOuiSGFJxoTfXePIqSYMJERRUVBcWkYWPSNOO1LCnXY12s6JpEv2sqqCKGzsgzl6TMo2BO/7ZUmxzNAK4xB+IkWcxvrq+v5nhx9/vV7Ts0RdsEwU/akJKlE5QaIhgpSWNYhb0sD/q/SG1FfWjRSlubgizJEUgQthaPpCCixqAnq0PDgS5Pdkkyu3l7t8DnIq6J1T5E3NglZ4XDayMG39fzuxme/Ta/fHd14pSKFRQ1nAijjw+4JuzToTYerVgxzg6Uh6dqKBHWdEYQztvL2zfH4WiyCR5MIIIKyw17gmKVyMV3UTHKSwQgpS5IQ/e7GVoTEDo3KfpOpXp2xjEqurQC0r9urDq1x5p9pOVFTVRJxWAXCPfPywVk/Cjewj7axmFtmFkhbauKFYwKgzaAqBB5IoyTJSTIPHfRqiX0uI5enK2Z2/68y6NVJFQiOMjNaWPXIfOrDzd379/hL5f7TJ1PF/jzlY1aqWWAEhI1ij4xaTUqlGxCdG+ufr18/3aBb6+uLxc3H67wbH73x2wB8WUBgHFutVGeLnLUECYME3UOlSQb/8/r65scEQXjAYkm3LWUNqxAhn50tJOjJbdKPec+SYpqyW3ASsMGJdMG6ALSHZoRoG0Ds0L6ohVRT8ANTqJkaWEypOCACSdqiAJxJSUwzJqoR2qQNoTzuIELxO1eyzwUyVmBK5UNpwj4yqwoeELvabv8ixYmTUZJkvwSeS8D2vtExXShLB0lXgTUFTZuyQsO1NLXxG/qEjyBOBT6B91KQb3QKFbXVGEjH6nwWi9uPAt2awHDAwNA8aFCU1pOgH5NWBna6G5ZW1YSR8owIpxOUAWcGzQbVppVZ7iirF6Zbq1C3NiReLeXooVcr6ko/dmwllYVrTpJSlqhpWW8xHU8fhtt1secHFG+174IH4BCYNRxPFaejNDFz19KbZtF6NAh6963m45d7h9iFXCzUkRTMO/P3wnj/XBgJwLzk3Mb/QbWAWBDCXSSC2xv1EqwtmvoueexgvSxJkvH+Y8oHfWKH8fN/VQpAQTCmR8jUgMzQJqAcGC2VtLIWpFmhWSFtm0qdjna9o66Q7WSbjaRo1PRjkGL3Eej4okpKaDKMB+CBCF33eEHe9u/EOLBHQtj89zQh10+gN4OCDna+4nDfVp0jpXkPMS4HWTyAHOf0kIKN8UTtG0lu2A1arsNQhfn+6ZLrdti2i99HlWD4Zy2q04d6jQNH534YFanZ+i0c3DNHiNwC8QqNFxrBM8zzxoIXlV0MCc9IDf+HZJbdcohGUSrobgz9wwRrfyqUwa+iNqw7NR9+pim0OC2cLRf4rZmad/0BKdMU2U5xVTUTFDMRBPvrtGAaUoG7uY0y5x8R34LrimpgU2di5vGewMXE72HM+aOCB8eYGzvw+wLCbcKh7u6xHGiO6pwgnEBowltX2ahaX2fQbft6WYb0/Tixba9Ibyqksr/kwcXN9RD4hvDW2Wts9Eu70GcepcG/c7/dageLw+7AOogoj3oJGL6PoUWJapYZapKs1c/fXdPLj69vPgBP4y2oKP+dZZ5nNEue9VTp/lxinrY/Vy7bxTQI1nWC28EvdC38SFql7WYItwdZG+1i5cAWAroxQwOpbIhyGjkTsapOJSj6RR9Hybw9A0OoHuX+5cP8HsI1IdI0/9CVV9BUWF1RFTD5jtPWHu7AzmCjP7fuWzAOF+msYH5SSJrH6v9R5Pql/3o1bT/5vwNmAwqwzQT8DKGnGXqs9/Ae4PZ9uspCo4Y/Yim/cXgdj73XvwamH8BUEsBAhQAFAAAAAgAAAAUXbY3XfUrAAAAKQAAAB0AAAAAAAAAAAAAAKSBAAAAAGluZmVyZW5jZV9hcGkvYXBwL19faW5pdF9fLnB5UEsBAhQAFAAAAAgAAAAUXfwFTasZAAAAFwAAACEAAAAAAAAAAAAAAKSBZgAAAGluZmVyZW5jZV9hcGkvYXBwL2FwaS9fX2luaXRfXy5weVBLAQIUABQAAAAIAAAAFF2muhuITgUAAGcRAAAfAAAAAAAAAAAAAACkgb4AAABpbmZlcmVuY2VfYXBpL2FwcC9hcGkvcm91dGVzLnB5UEsBAhQAFAAAAAgAAAAUXY4BlP8uAAAALAAAACIAAAAAAAAAAAAAAKSBSQYAAGluZmVyZW5jZV9hcGkvYXBwL2NvcmUvX19pbml0X18ucHlQSwECFAAUAAAACAAAABRdel84y/cDAAAhCgAAIAAAAAAAAAAAAAAApIG3BgAAaW5mZXJlbmNlX2FwaS9hcHAvY29yZS9jb25maWcucHlQSwECFAAUAAAACAAAABRdUGTd89QAAADWAQAAIAAAAAAAAAAAAAAApIHsCgAAaW5mZXJlbmNlX2FwaS9hcHAvY29yZS9lcnJvcnMucHlQSwECFAAUAAAACAAAABRdNRRT2jMGAADfEgAAGQAAAAAAAAAAAAAApIH+CwAAaW5mZXJlbmNlX2FwaS9hcHAvbWFpbi5weVBLAQIUABQAAAAIAAAAFF11+wteLAAAACoAAAAkAAAAAAAAAAAAAACkgWgSAABpbmZlcmVuY2VfYXBpL2FwcC9tb2RlbHMvX19pbml0X18ucHlQSwECFAAUAAAACAAAABRdITkg2K8EAAD7DQAAIwAAAAAAAAAAAAAApIHWEgAAaW5mZXJlbmNlX2FwaS9hcHAvbW9kZWxzL3NjaGVtYXMucHlQSwECFAAUAAAACAAAABRdkNkzMCQAAAAiAAAAJwAAAAAAAAAAAAAApIHGFwAAaW5mZXJlbmNlX2FwaS9hcHAvcHJvdmlkZXJzL19faW5pdF9fLnB5UEsBAhQAFAAAAAgAAAAUXaA6DlVWAQAA0wIAACMAAAAAAAAAAAAAAKSBLxgAAGluZmVyZW5jZV9hcGkvYXBwL3Byb3ZpZGVycy9iYXNlLnB5UEsBAhQAFAAAAAgAAAAUXW2EgEjfBAAAKBAAADEAAAAAAAAAAAAAAKSBxhkAAGluZmVyZW5jZV9hcGkvYXBwL3Byb3ZpZGVycy9kaWZmdXNlcnNfcHJvdmlkZXIucHlQSwECFAAUAAAACAAAABRdbZh2DWEFAABCEAAALgAAAAAAAAAAAAAApIH0HgAAaW5mZXJlbmNlX2FwaS9hcHAvcHJvdmlkZXJzL2xvcmFfdmFsaWRhdGlvbi5weVBLAQIUABQAAAAIAAAAFF3holEYWQQAAC4NAAAjAAAAAAAAAAAAAACkgaEkAABpbmZlcmVuY2VfYXBpL2FwcC9wcm92aWRlcnMvbW9jay5weVBLAQIUABQAAAAIAAAAFF18OU+KMAAAAC4AAAAmAAAAAAAAAAAAAACkgTspAABpbmZlcmVuY2VfYXBpL2FwcC9zZXJ2aWNlcy9fX2luaXRfXy5weVBLAQIUABQAAAAIAAAAFF1/+OlV/AQAAFwRAAAwAAAAAAAAAAAAAACkga8pAABpbmZlcmVuY2VfYXBpL2FwcC9zZXJ2aWNlcy9nZW5lcmF0aW9uX3NlcnZpY2UucHlQSwECFAAUAAAACAAAABRdsmDg+5EFAADIEAAALAAAAAAAAAAAAAAApIH5LgAAaW5mZXJlbmNlX2FwaS9hcHAvc2VydmljZXMvcHJvbXB0X2J1aWxkZXIucHlQSwUGAAAAABEAEQB7BQAA1DQAAAAA'
RUNTIME_SOURCE_MANIFEST = '{"inference_api/app/__init__.py": "abb5fab205e56925415c22f6377905d558305c659b5db00af87f6961c6eb2618", "inference_api/app/api/__init__.py": "d93b5f927d5548a8ac36fcb08ff9ea4d61844066f86244c9564b8b01c01428b4", "inference_api/app/api/routes.py": "f856ab51080e273cb438c5e767209667807df1982a8549a29a07ba00418539f0", "inference_api/app/core/__init__.py": "cbfe6b89cb869af3e65ac9a63c362a967caabea9dd46246b5cf8830e15af07be", "inference_api/app/core/config.py": "d8e6afcea6714e44fb92fd9bd0a3ee229f08c0b37385d42ca9e282803cbfc0b5", "inference_api/app/core/errors.py": "6dd9efe972f0c7f5313fb893aaa91570f3b3b2eb7d388a4e3b9b969940795c7f", "inference_api/app/main.py": "a515f99b240d34b6465e46a3d5cde4cc15513b03e11b85755e5895f301472998", "inference_api/app/models/__init__.py": "0898b02a1665e1da76b79fb770f79de3ab4281dcd566651ba4b9f3e8d52f3920", "inference_api/app/models/schemas.py": "df496a843636f47580f314d0f5860dd7352d290db3cdc0647a4ad3aa1ba87dc8", "inference_api/app/providers/__init__.py": "ea78abe5924b9ee4c67f896ce11db4bf0050cbd5b490395e5eef7bd3182e931e", "inference_api/app/providers/base.py": "187920a833aec96b05d360c80949e2f573b4dac920c08bb945d85f08ed3eb84f", "inference_api/app/providers/diffusers_provider.py": "c57e888b2929dde7952af3cd23a086f5d92a892e6c4b83077d49ccf913b5ed89", "inference_api/app/providers/lora_validation.py": "ea16e804b4c0f207add5d7dcf1e247cdfe3dc1af030c725540ab25ddf26c0367", "inference_api/app/providers/mock.py": "7d5c71bbb7ba62138d97fd88c106931fcdcba6be1f4e14a8235b44cfcdb26a3f", "inference_api/app/services/__init__.py": "ea99f2f0f3b51e94912e9161c705257c74e5a8faf58f44e05b5517515d9e2373", "inference_api/app/services/generation_service.py": "5a2826aed68df164b6046b789f99f0d1f7767d558c19689920aca98faabcf8fd", "inference_api/app/services/prompt_builder.py": "11977ed5ed7ecee6b15f51625cc6e952e0e21f5f6a8b61dc74ef03f591ed140f"}'
RUNTIME_SOURCE_MANIFEST = json.loads(RUNTIME_SOURCE_MANIFEST)

archive_payload = base64.b64decode(RUNTIME_ARCHIVE_B64, validate=True)
if hashlib.sha256(archive_payload).hexdigest() != RUNTIME_ARCHIVE_SHA256:
    raise SystemExit("STOP: embedded inference_api archive hash mismatch.")
if RUNTIME_ROOT.exists():
    shutil.rmtree(RUNTIME_ROOT)
RUNTIME_ROOT.mkdir(parents=True)
with zipfile.ZipFile(io.BytesIO(archive_payload)) as archive:
    unsafe = [name for name in archive.namelist() if Path(name).is_absolute() or ".." in Path(name).parts]
    if unsafe:
        raise SystemExit(f"STOP: unsafe embedded runtime paths: {unsafe}")
    archive.extractall(RUNTIME_ROOT)

for relative, expected in RUNTIME_SOURCE_MANIFEST.items():
    extracted = RUNTIME_ROOT / relative
    if not extracted.is_file() or sha256_file(extracted) != expected:
        raise SystemExit(f"STOP: embedded runtime source mismatch: {relative}")
sys.path.insert(0, str(RUNTIME_ROOT))
print("Embedded inference_api snapshot: PASS")
print("Archive SHA-256:", RUNTIME_ARCHIVE_SHA256)
print("Source files:", len(RUNTIME_SOURCE_MANIFEST))

In [ ]:
os.environ.update(
    {
        "INFERENCE_PROVIDER": "diffusers",
        "BASE_MODEL": BASE_MODEL,
        "LORA_PATH": str(LORA_PATH),
        "LORA_SHA256": LORA_SHA256,
        "LORA_IDENTIFIER": LORA_IDENTIFIER,
        "REQUIRE_GPU": "true",
        "INFERENCE_REQUIRE_TOKEN": "true",
        "INFERENCE_API_TOKEN": INFERENCE_API_TOKEN,
        "GENERATED_DIR": str(GENERATED_DIR),
        "CORS_ORIGINS": "",
        "GENERATION_TIMEOUT_SECONDS": "85",
        "GENERATION_MAX_RETRIES": "1",
        "DEFAULT_SEED": "42",
    }
)

from inference_api.app.core.config import Settings
from inference_api.app.main import create_app
from inference_api.app.providers.lora_validation import validate_lora_artifact
from inference_api.app.services.generation_service import GenerationService

SETTINGS = Settings(_env_file=None)
LORA_INFO = validate_lora_artifact(SETTINGS.lora_path, SETTINGS.lora_sha256)
INFERENCE_SERVICE = GenerationService(SETTINGS)
MODEL_HEALTH = INFERENCE_SERVICE.health()
if not (
    MODEL_HEALTH.get("provider") == "diffusers"
    and MODEL_HEALTH.get("device") == "cuda"
    and MODEL_HEALTH.get("model_loaded") is True
    and MODEL_HEALTH.get("lora_sha256") == LORA_SHA256
):
    raise SystemExit("STOP: model/LoRA gagal dimuat: " + json.dumps(MODEL_HEALTH, indent=2))

FASTAPI_APP = create_app(SETTINGS, INFERENCE_SERVICE)
print("Model loaded once: PASS")
print(json.dumps(MODEL_HEALTH, indent=2))

In [ ]:
import threading
import uvicorn


def request_json(url: str, *, token: str = "", method: str = "GET", body=None, timeout=30):
    headers = {"Accept": "application/json", "User-Agent": "TaniAdaptif-Sidang/1.0"}
    if token:
        headers["X-Inference-Token"] = token
    payload = None
    if body is not None:
        headers["Content-Type"] = "application/json"
        payload = json.dumps(body).encode("utf-8")
    request = urllib.request.Request(url, data=payload, headers=headers, method=method)
    with urllib.request.urlopen(request, timeout=timeout) as response:
        return json.loads(response.read().decode("utf-8"))


UVICORN_CONFIG = uvicorn.Config(
    FASTAPI_APP,
    host=FASTAPI_HOST,
    port=FASTAPI_PORT,
    log_level="info",
    access_log=False,
    proxy_headers=True,
    forwarded_allow_ips="127.0.0.1",
)
UVICORN_SERVER = uvicorn.Server(UVICORN_CONFIG)
UVICORN_THREAD = threading.Thread(target=UVICORN_SERVER.run, daemon=True)
UVICORN_THREAD.start()

LOCAL_URL = f"http://{FASTAPI_HOST}:{FASTAPI_PORT}"
deadline = time.time() + 30
while time.time() < deadline:
    try:
        local_health = request_json(f"{LOCAL_URL}/health", timeout=3)
        if local_health.get("status") == "ok":
            break
    except Exception:
        time.sleep(1)
else:
    raise SystemExit("STOP: FastAPI gagal aktif pada port 5000.")

local_ready = request_json(f"{LOCAL_URL}/ready", token=INFERENCE_API_TOKEN, timeout=10)
if local_ready.get("status") != "ready":
    raise SystemExit("STOP: FastAPI aktif tetapi provider tidak ready.")
print("FastAPI local health: PASS")
print("FastAPI port:", FASTAPI_PORT)

In [ ]:
import queue

if "CLOUDFLARED_PROCESS" in globals() and CLOUDFLARED_PROCESS.poll() is None:
    CLOUDFLARED_PROCESS.terminate()
    CLOUDFLARED_PROCESS.wait(timeout=10)

cloudflared_home = WORK_ROOT / "cloudflared_home"
cloudflared_home.mkdir(parents=True, exist_ok=True)
cloudflared_env = {**os.environ, "HOME": str(cloudflared_home)}
CLOUDFLARED_PROCESS = subprocess.Popen(
    [
        str(CLOUDFLARED_PATH),
        "tunnel",
        "--url",
        LOCAL_URL,
        "--no-autoupdate",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=cloudflared_env,
)

tunnel_lines = queue.Queue()


def collect_tunnel_output():
    assert CLOUDFLARED_PROCESS.stdout is not None
    for line in CLOUDFLARED_PROCESS.stdout:
        tunnel_lines.put(line.rstrip())


threading.Thread(target=collect_tunnel_output, daemon=True).start()
tunnel_pattern = re.compile(r"https://[a-z0-9-]+\.trycloudflare\.com")
KAGGLE_GPU_URL = ""
deadline = time.time() + 90
while time.time() < deadline and CLOUDFLARED_PROCESS.poll() is None:
    try:
        line = tunnel_lines.get(timeout=1)
    except queue.Empty:
        continue
    match = tunnel_pattern.search(line)
    if match:
        KAGGLE_GPU_URL = match.group(0)
        break
if not KAGGLE_GPU_URL:
    raise SystemExit("STOP: Cloudflare Quick Tunnel gagal menghasilkan URL HTTPS.")

deadline = time.time() + 90
while time.time() < deadline:
    try:
        public_health = request_json(f"{KAGGLE_GPU_URL}/health", timeout=10)
        if public_health.get("status") == "ok":
            break
    except Exception:
        time.sleep(2)
else:
    raise SystemExit("STOP: URL Quick Tunnel tersedia tetapi health publik gagal.")

public_ready = request_json(
    f"{KAGGLE_GPU_URL}/ready",
    token=INFERENCE_API_TOKEN,
    timeout=20,
)
if public_ready.get("status") != "ready":
    raise SystemExit("STOP: readiness publik bukan ready.")
print(f"KAGGLE_GPU_URL={KAGGLE_GPU_URL}")
print("Cloudflare tunnel health: PASS")

In [ ]:
SANITY_REQUEST_ID = "kaggle-sidang-sanity-42"
sanity_response = request_json(
    f"{KAGGLE_GPU_URL}/v1/generate",
    token=INFERENCE_API_TOKEN,
    method="POST",
    body={
        "crop": "padi",
        "land_type": "sawah",
        "water_availability": "cukup",
        "previous_crop": "jagung",
        "weather_summary": "Cerah berawan; suhu 28 C; kelembaban 78 persen",
        "seed": 42,
        "request_id": SANITY_REQUEST_ID,
        "steps": 15,
        "guidance_scale": 7.5,
        "width": 512,
        "height": 512,
    },
    timeout=100,
)
metadata = sanity_response.get("metadata") or {}
sanity_checks = {
    "success": sanity_response.get("success") is True,
    "request_id": sanity_response.get("request_id") == SANITY_REQUEST_ID,
    "model": sanity_response.get("model") == BASE_MODEL,
    "provider": metadata.get("provider") == "diffusers",
    "device": metadata.get("device") == "cuda",
    "not_mock": metadata.get("is_mock") is False,
    "lora_sha256": metadata.get("lora_sha256") == LORA_SHA256,
}
if not all(sanity_checks.values()):
    raise SystemExit("STOP: sanity metadata gagal: " + json.dumps(sanity_checks, indent=2))

image_path_from_api = urllib.parse.urlsplit(str(sanity_response.get("image_url") or "")).path
if not image_path_from_api.startswith("/static/generated/"):
    raise SystemExit("STOP: sanity image_url tidak menggunakan static/generated.")
public_image_url = urllib.parse.urljoin(KAGGLE_GPU_URL, image_path_from_api)
image_request = urllib.request.Request(
    public_image_url,
    headers={
        "Accept": "image/png",
        "User-Agent": "TaniAdaptif-Sidang/1.0",
        "X-Inference-Token": INFERENCE_API_TOKEN,
    },
)
with urllib.request.urlopen(image_request, timeout=30) as response:
    image_content_type = response.headers.get_content_type()
    image_bytes = response.read()
if image_content_type != "image/png" or image_bytes[:8] != b"\x89PNG\r\n\x1a\n":
    raise SystemExit("STOP: sanity artifact bukan PNG valid.")
if len(image_bytes) < 24 or image_bytes[12:16] != b"IHDR":
    raise SystemExit("STOP: sanity PNG tidak memiliki IHDR valid.")
width = int.from_bytes(image_bytes[16:20], "big")
height = int.from_bytes(image_bytes[20:24], "big")
if (width, height) != (512, 512):
    raise SystemExit(f"STOP: dimensi sanity PNG salah: {width}x{height}")

SANITY_PATH = WORK_ROOT / "sanity_inference.png"
SANITY_PATH.write_bytes(image_bytes)
SANITY_SHA256 = hashlib.sha256(image_bytes).hexdigest()
print("Sanity inference PNG: PASS")
print("Sanity path:", SANITY_PATH)
print("Sanity bytes:", len(image_bytes))
print("Sanity SHA-256:", SANITY_SHA256)

In [ ]:
model_info = request_json(
    f"{KAGGLE_GPU_URL}/v1/model-info",
    token=INFERENCE_API_TOKEN,
    timeout=20,
)
if not (
    model_info.get("base_model") == BASE_MODEL
    and model_info.get("lora_identifier") == LORA_IDENTIFIER
    and model_info.get("provider") == "diffusers"
):
    raise SystemExit("STOP: model-info tidak sesuai model final.")

STATE_PATH = WORK_ROOT / "kaggle_live_inference_state.json"
state = {
    "status": "success",
    "training_performed": False,
    "gpu": gpu.name,
    "fastapi_port": FASTAPI_PORT,
    "kaggle_gpu_url": KAGGLE_GPU_URL,
    "health": public_health,
    "ready": public_ready,
    "base_model": BASE_MODEL,
    "lora_identifier": LORA_IDENTIFIER,
    "lora_sha256": LORA_SHA256,
    "sanity_inference": {
        "status": "success",
        "path": str(SANITY_PATH),
        "sha256": SANITY_SHA256,
        "bytes": len(image_bytes),
        "width": width,
        "height": height,
    },
    "secret_names": ["INFERENCE_API_TOKEN", "HF_TOKEN (optional)"],
    "secret_values_recorded": False,
}
STATE_PATH.write_text(json.dumps(state, indent=2) + "\n", encoding="utf-8")

print("GPU=" + gpu.name)
print("FASTAPI_PORT=5000")
print("SECRET_NAMES=INFERENCE_API_TOKEN,HF_TOKEN(optional)")
print("KAGGLE_GPU_URL=" + KAGGLE_GPU_URL)
print("HEALTH=PASS")
print("SANITY_INFERENCE=PASS")
print("STATUS=READY_FOR_SIDANG")

In [ ]:
KEEP_ALIVE_MINUTES = 330
print(f"Service live selama maksimal {KEEP_ALIVE_MINUTES} menit. Biarkan cell ini berjalan saat sidang.")
for elapsed_minutes in range(KEEP_ALIVE_MINUTES):
    if not UVICORN_THREAD.is_alive():
        raise RuntimeError("FastAPI thread berhenti.")
    if CLOUDFLARED_PROCESS.poll() is not None:
        raise RuntimeError("Cloudflare tunnel berhenti.")
    if elapsed_minutes % 15 == 0:
        check = request_json(f"{LOCAL_URL}/health", timeout=5)
        if check.get("status") != "ok":
            raise RuntimeError("FastAPI health tidak lagi ok.")
        print(f"KEEP_ALIVE minute={elapsed_minutes} KAGGLE_GPU_URL={KAGGLE_GPU_URL}")
    time.sleep(60)
print("Keep-alive selesai. Run ulang notebook untuk membuat tunnel baru.")